# Notebook da Camada Bronze — `Landing_to_Bronze`

Tarefa `to_Bronze` do Job — Projeto CineData Analytics (Atividade 3)

Este notebook faz a ingestão dos dados brutos de três fontes:
- **CSVs da base CineData:** os cinco arquivos do volume `landing` viram uma tabela Delta cada no schema `bronze`.
- **API PTAX do Banco Central:** a cotação do dólar do período informado nos widgets (por padrão, os últimos 7 dias) é gravada em `bronze.tb_cotacao_dolar`.
- **`title.basics.tsv` do IMDb:** a coluna de gêneros é gravada em `bronze.genres_imdb`, como referência para validar os gêneros na Silver.

## Mudanças na Atividade 3
- **Widgets prontos para o Job:** os widgets agora são criados antes de serem lidos. Na versão anterior, a leitura vinha primeiro e só funcionava porque os widgets já existiam de execuções manuais. Uma tarefa do Job começa num contexto limpo, sem widgets, e a leitura falharia (`InputWidgetNotDefined`).
- **Período calculado a cada execução:** o padrão (hoje − 7 dias até hoje) é calculado no momento da execução. No Job, os widgets são criados do zero a cada run, então o período acompanha a data da execução. Por isso, `data_inicio` e `data_fim` não são configurados como parâmetros da tarefa: um valor fixo ali substituiria o padrão em todas as execuções. No uso interativo, um widget já criado mantém o valor anterior até ser editado ou removido.
- **Nova fonte: gêneros do IMDb.** O arquivo `title.basics.tsv` vem dos datasets públicos do IMDb (uso pessoal e não comercial). Só a coluna `genres` é gravada, exatamente como vem, com o marcador de ausência `\N` e as listas separadas por vírgula.
- **Saídas de inspeção comentadas:** os `print` e `display` usados no desenvolvimento estão comentados. Cada um dispara uma leitura completa da tabela, o que só atrasaria o Job. Para depurar, basta descomentar.

## Decisões adotadas
- **Dado cru, sem alteração:** todas as colunas são lidas como texto, sem `inferSchema`. Valores como `"Unknown"` no meio de colunas numéricas são preservados como vieram, e a tipagem e a limpeza ficam para a Silver.
- **Leitura robusta de CSV:** as opções `multiLine` e `escape` tratam textos longos (sinopses, comentários) com vírgulas, aspas e quebras de linha internas, que de outra forma quebrariam as linhas.
- **Leitura do TSV do IMDb:** separador `\t` e tratamento de aspas desligado (`quote = ""`). O IMDb não usa aspas como delimitador, e títulos que contêm `"` desalinhariam as colunas no parser padrão.
- **Padrão de nome de arquivo:** a leitura usa curinga (`*movies_info*.csv`) porque os nomes dos arquivos têm prefixos numéricos e sufixos inconsistentes (`TMDB_IMDB` / `IMDB_TMDB`).
- **Rastreabilidade:** cada tabela recebe a coluna `ingestion_datetime`, com o momento exato da carga. O valor é o mesmo para todas as linhas de uma carga, o que permite à Silver identificar o lote mais recente.
- **Modo `append`:** cada execução do Job acrescenta uma nova cópia completa de cada arquivo, sem apagar as cargas anteriores. As duplicatas são intencionais: a Bronze guarda o histórico do que chegou e quando, e a Silver usa o `ingestion_datetime` para chegar ao estado atual. Em consequência, a Bronze cresce a cada execução. Em produção, isso seria controlado com ingestão incremental (`COPY INTO` ou Auto Loader, que ignoram arquivos já carregados) ou com uma política de retenção (`DELETE` dos lotes antigos seguido de `VACUUM`).
- **API do Banco Central:**
  - As datas vêm de widgets no formato `MM-DD-AAAA`, exigido pela API, e são validadas antes da chamada.
  - O padrão de 7 dias corridos garante que haja dias úteis no período, já que a API não retorna cotação em fins de semana e feriados.
  - A chamada tem timeout de 30 segundos, e `raise_for_status()` interrompe a execução se a API responder com erro HTTP. No Job, essa falha impede que `to_Silver` rode.
  - Se o período não tiver cotações, nada é gravado e o notebook apenas avisa. A Silver continua usando o histórico de cotações já acumulado.

In [0]:
#parametros do ambiente

from pyspark.sql import functions as F
from datetime import date, timedelta, datetime
import requests

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "cinedata_medallion"
bronze_schema = f"{catalog}.bronze"
volume_path = f"/Volumes/{catalog}/bronze/landing"


# Landing -> Bronze: coluna genres do title.basics (IMDb), do jeito que vem na origem
# Fonte: https://developer.imdb.com/non-commercial-datasets (uso pessoal e não comercial)
caminho_title_basics = f"{volume_path}/title.basics.tsv"
tabela_generos = f"{bronze_schema}.genres_imdb"

# Tabela de cotacao do dólar
tabela_cotacao = f"{catalog}.bronze.tb_cotacao_dolar"

# Tabelas Bronze a partir dos CSVs
tabelas = ["movies_info", "movies_financials", "movies_metrics",
           "credits_and_tags", "movies_reviews"]

#variavel do dia atual 
hoje = date.today()


In [0]:
#Este comando foi adicionado para limpar os widgets do notebook e verificar o funcionamento deles
#dbutils.widgets.removeAll()

In [0]:
# Esta celula serve para definição dos parametros de data do notebook (usado na extração da API)
# Eles podem ser modificados diretamente no cabeçalho do notebook

# definindo as datas de hoje e de 7 dias atrás
#print(f"Data de hoje: {hoje}")
padrao_inicio = (hoje - timedelta(days=7)).strftime("%m-%d-%Y")
padrao_fim = hoje.strftime("%m-%d-%Y")

# definindo os widgets
dbutils.widgets.text("data_inicio", padrao_inicio, "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", padrao_fim, "Data fim (MM-DD-AAAA)")


# datas vindas dos widgets (formato MM-DD-AAAA)
data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")


# imprime os widgets (na atividade 2 estes prints estavam acima da definição dos widgets, o que funcionava se os widgets ja estavam definidos, mas gerava erro na primeira execução)
# Após setados os widgets não mudam em outras execuções, mas dentro de cada job o widget é atualizado
# print("Definindo widgets:")
# print(f"Data início: {data_inicio_formatada}")
# print(f"Data fim: {data_fim_formatada}")

## Criação das tabelas no schema Bronze
Esta célula roda um for, de tabela a tabela carregando os dados contidos nos csvs

In [0]:
# Loop de criação das tabelas
for tabela in tabelas:
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)
          .option("quote", '"')
          .option("escape", '"')
          .csv(f"{volume_path}/*{tabela}*.csv")
          .withColumn("ingestion_datetime", F.current_timestamp()))


    (df.write     # escreve a tabela como delta em modo append
       .format("delta")
       .mode("append")
       .saveAsTable(f"{bronze_schema}.{tabela}"))

    #print(f"{bronze_schema}.{tabela}: {df.count()} linhas lidas")

## Extração e Criação da tabela de Cotação
Esta célula faz a requisição para a API do Banco Central com as cotações, armazena em um df se obtiver resposta e carrega na tabela designada

In [0]:
# TRUNCATE TABLE caso desejado
#spark.sql(f"TRUNCATE TABLE {bronze_schema}.tb_cotacao_dolar")
#print(f"Tabela {bronze_schema}.tb_cotacao_dolar truncada com sucesso.")

In [0]:
# valida o formato antes de chamar a API (erro aqui = data digitada errada)
for d in (data_inicio_formatada, data_fim_formatada):
    datetime.strptime(d, "%m-%d-%Y")

# endpoint PTAX do Banco Central: cotações do período, só data/hora e valor de compra
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'"
    f"&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

# chamada à API; para a execução se a resposta for erro HTTP
resposta = requests.get(url, timeout=30)
resposta.raise_for_status()
cotacoes = resposta.json()["value"]

print(f"Período: {data_inicio_formatada} a {data_fim_formatada} | {len(cotacoes)} cotações retornadas")

# grava só se houver cotações (período só com fim de semana/feriado volta vazio)
if cotacoes:
    df = (spark.createDataFrame(cotacoes)
          .select("dataHoraCotacao", "cotacaoCompra")
          .withColumn("ingestion_datetime", F.current_timestamp()))  # timestamp da carga

    # bronze em append: cada execução acrescenta, sem apagar cargas anteriores
    (df.write
       .format("delta")
       .mode("append")
       .saveAsTable(tabela_cotacao))

    #display(spark.table(tabela_cotacao).orderBy(F.desc("ingestion_datetime")))
else:
    print("Nenhuma cotação no período (fim de semana/feriado?). Nada gravado.")

In [0]:

# desalinham as colunas e o genres de algumas linhas viria de outra coluna
df_genres_imdb = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("quote", "")
    .csv(caminho_title_basics)
    .select("genres")                                         # sem tratamento: \N e listas com vírgula ficam como vieram
    .withColumn("ingestion_datetime", F.current_timestamp())
)

# Append, seguindo o padrão das demais tabelas Bronze (histórico de cargas)
(df_genres_imdb.write
    .format("delta")
    .mode("append")
    .saveAsTable(tabela_generos))

# Uma linha por carga: mostra quantos registros entraram em cada execução
# display(
#     spark.table(tabela_generos)
#     .orderBy(F.col("ingestion_datetime").desc())
# )

In [0]:
#numero de rows em cada tabela
# tabelas = spark.catalog.listTables(bronze_schema)
# for t in tabelas:
#   print(f"{t.name}: {spark.table(f'{bronze_schema}.{t.name}').count()} linhas")
  

In [0]:
# Celula adicional mostrando todas as tabelas da bronze carregadas (serviu de apoio para as transformações feitas na camada silver)

#for t in tabelas:
#    display(spark.table(f"{bronze_schema}.{t.name}"))                   